In [ ]:
!pip install ultralytics
!pip install roboflow

In [ ]:
!pip install grad-cam

In [ ]:
import torch
import os 
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import cv2 
import yaml 
import glob 
import numpy as np
import matplotlib.pyplot as plt 
import random
import torch.nn as nn

from ultralytics import YOLO
from PIL import Image
from roboflow import Roboflow
from torchvision import transforms
from skimage.transform import resize
from tqdm import tqdm

In [ ]:
rf = Roboflow(api_key="sNwZKFyXE0vHSdAqPXX0")
project = rf.workspace("main-workspace-vcien").project("birdnest-balanced-classification")
version = project.version(3)
dataset = version.download("folder")

In [ ]:
model = YOLO("yolo11m-cls.pt")

In [ ]:
model.train(
    data="BirdNest-Balanced-Classification-3", 
    epochs=30, 
    patience=10,
    imgsz=224
)

In [ ]:
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")

In [ ]:
metrics = loaded_model.val(data="BirdNest-Balanced-Classification-3", split="test", plots=True)

In [ ]:
print(metrics.results_dict)

In [ ]:
top1_accuracy = metrics.results_dict["metrics/accuracy_top1"]

print(f"Model Accuracy: {top1_accuracy}")

# Prediction Logits & Probabilities

In [ ]:
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
image_path = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"
img = Image.open(image_path)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
results = loaded_model.predict(
    source=image_path,
    verbose=False
)

res = results[0]

probs = res.probs.data.cpu().numpy()
class_names = loaded_model.names

print("Probabilities:")
for i, p in enumerate(probs):
    print(f"  {class_names[i]}: {p:.4f}")

print(
    "\nPredicted class:",
    class_names[res.probs.top1],
    f"(confidence={res.probs.top1conf:.4f})"
)

# GradCAM

In [ ]:
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
image_path = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"
img = Image.open(image_path)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Load model
model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
torch_model = model.model

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_model = torch_model.to(device)

for param in torch_model.parameters():
    param.requires_grad = True

torch_model.eval()

# 2. Target the last spatial layer (C2PSA at index -2)
target_layers = [torch_model.model[-2]]

# 3. Load and preprocess image
img_path = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"

# First get YOLO's correct prediction
results = model.predict(source=img_path, verbose=False)
predicted_class = results[0].probs.top1
confidence = results[0].probs.top1conf.item()
class_names = model.names

print(f"Predicted class: {predicted_class} ({class_names[predicted_class]})")
print(f"Confidence: {confidence:.4f}")

# Preprocess using YOLO's method
from ultralytics.data.augment import classify_transforms

img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Convert to PIL Image for transforms
img_pil = Image.fromarray(img_rgb)

# Use YOLO's preprocessing pipeline
transform = classify_transforms(size=224)
img_tensor = transform(img_pil)
input_tensor = img_tensor.unsqueeze(0).to(device)
input_tensor.requires_grad = True

# For visualization overlay
img_resized = cv2.resize(img_rgb, (224, 224))
rgb_img = np.float32(img_resized) / 255

# 4. Verify preprocessing matches YOLO
with torch.no_grad():
    output = torch_model(input_tensor)
    if isinstance(output, tuple):
        output = output[0]
    manual_probs = torch.softmax(output, dim=1)
    manual_pred = torch.argmax(output).item()
    
    print(f"\nVerification:")
    print(f"Manual preprocessing predicts: {manual_pred} (prob: {manual_probs[0][manual_pred]:.4f})")
    print(f"YOLO .predict() predicts: {predicted_class} (prob: {confidence:.4f})")
    print(f"Match: {manual_pred == predicted_class}")

# 5. Initialize Grad-CAM
cam = GradCAM(model=torch_model, target_layers=target_layers)

# 6. Generate Grad-CAM for the PREDICTED class
targets = [ClassifierOutputTarget(predicted_class)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

# 7. Overlay
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img_resized)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM - Class {predicted_class} ({class_names[predicted_class]})\nConfidence: {confidence:.4f}")
plt.imshow(visualization)
plt.axis('off')

plt.tight_layout()
plt.savefig('bird_nest_gradcam.jpg', dpi=150, bbox_inches='tight')
plt.show()

print("\nGrad-CAM completed successfully.")

# LIME

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from lime import lime_image
from skimage.segmentation import mark_boundaries

# 1. Load model
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
image_path = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_model = loaded_model.model.to(device)
torch_model.eval()

# Get class names
class_names = loaded_model.names

# 2. Create prediction function for LIME
def predict_fn(images):
    """
    Prediction function for LIME.
    Input: numpy array of images (N, H, W, 3) in range [0, 255]
    Output: numpy array of probabilities (N, num_classes)
    """
    from ultralytics.data.augment import classify_transforms
    
    batch_predictions = []
    transform = classify_transforms(size=224)
    
    for img in images:
        # Convert numpy array to PIL Image
        img_uint8 = img.astype(np.uint8)
        img_pil = Image.fromarray(img_uint8)
        
        # Apply YOLO's preprocessing
        img_tensor = transform(img_pil)
        input_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Get prediction
        with torch.no_grad():
            output = torch_model(input_tensor)
            if isinstance(output, tuple):
                output = output[0]
            probs = torch.softmax(output, dim=1).cpu().numpy()[0]
        
        batch_predictions.append(probs)
    
    return np.array(batch_predictions)

# 3. Load and prepare the image
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Get the true prediction from YOLO
results = loaded_model.predict(source=image_path, verbose=False)
predicted_class = results[0].probs.top1
confidence = results[0].probs.top1conf.item()

print(f"Predicted class: {predicted_class} ({class_names[predicted_class]})")
print(f"Confidence: {confidence:.4f}")

# 4. Initialize LIME explainer
explainer = lime_image.LimeImageExplainer()

# 5. Generate LIME explanation
print("\nGenerating LIME explanation (this may take a minute)...")
explanation = explainer.explain_instance(
    img_rgb,
    predict_fn,
    top_labels=3,  # Explain top 3 classes
    hide_color=0,  # Color to use for hidden superpixels
    num_samples=1000  # Number of perturbed samples (higher = more accurate but slower)
)

# 6. Get the explanation for the predicted class
temp, mask = explanation.get_image_and_mask(
    predicted_class,
    positive_only=True,  # Show only positive contributions
    num_features=10,  # Number of superpixels to highlight
    hide_rest=False  # Whether to hide non-important regions
)

# 7. Create visualization with boundaries
img_boundary = mark_boundaries(temp / 255.0, mask)

# Also create heatmap version (positive and negative)
temp_full, mask_full = explanation.get_image_and_mask(
    predicted_class,
    positive_only=False,
    num_features=10,
    hide_rest=False
)
img_boundary_full = mark_boundaries(temp_full / 255.0, mask_full)

# 8. Display results
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Original image
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

# LIME explanation (positive only)
axes[0, 1].imshow(img_boundary)
axes[0, 1].set_title(f"LIME Explanation (Positive)\nClass {predicted_class} ({class_names[predicted_class]})\nConf: {confidence:.4f}")
axes[0, 1].axis('off')

# LIME explanation (positive and negative)
axes[1, 0].imshow(img_boundary_full)
axes[1, 0].set_title("LIME Explanation (Pos + Neg)")
axes[1, 0].axis('off')

# Heatmap overlay
from matplotlib.colors import LinearSegmentedColormap
heatmap = np.zeros(mask.shape)
for i in np.unique(mask):
    if i != 0:  # Skip background
        heatmap[mask == i] = 1

axes[1, 1].imshow(img_rgb)
axes[1, 1].imshow(heatmap, cmap='jet', alpha=0.5)
axes[1, 1].set_title("LIME Heatmap Overlay")
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('bird_nest_lime.jpg', dpi=150, bbox_inches='tight')
plt.show()

# 9. Print feature importance
print("\n" + "="*60)
print("LIME Feature Importance:")
print("="*60)
local_exp = explanation.local_exp[predicted_class]
for feature, weight in sorted(local_exp, key=lambda x: abs(x[1]), reverse=True)[:10]:
    contribution = "positive" if weight > 0 else "negative"
    print(f"Superpixel {feature}: {weight:.4f} ({contribution})")

print("\nLIME completed successfully!")

# RISE 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from skimage.transform import resize
from tqdm import tqdm


class RISE(nn.Module):
    def __init__(self, model, input_size, gpu_batch=100):
        super(RISE, self).__init__()
        self.model = model
        self.input_size = input_size
        self.gpu_batch = gpu_batch

    def generate_masks(self, N, s, p1, savepath='masks.npy'):
        cell_size = np.ceil(np.array(self.input_size) / s)
        up_size = (s + 1) * cell_size

        grid = np.random.rand(N, s, s) < p1
        grid = grid.astype('float32')

        self.masks = np.empty((N, *self.input_size))

        for i in tqdm(range(N), desc='Generating filters'):
            # Random shifts
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            # Linear upsampling and cropping
            self.masks[i, :, :] = resize(grid[i], up_size, order=1, mode='reflect',
                                         anti_aliasing=False)[x:x + self.input_size[0], y:y + self.input_size[1]]
        self.masks = self.masks.reshape(-1, 1, *self.input_size)
        np.save(savepath, self.masks)
        self.masks = torch.from_numpy(self.masks).float()
        self.masks = self.masks.cuda()
        self.N = N
        self.p1 = p1

    def load_masks(self, filepath):
        self.masks = np.load(filepath)
        self.masks = torch.from_numpy(self.masks).float().cuda()
        self.N = self.masks.shape[0]

    def forward(self, x):
        N = self.N
        _, _, H, W = x.size()
        # Apply array of filters to the image
        stack = torch.mul(self.masks, x.data)

        # p = nn.Softmax(dim=1)(model(stack)) processed in batches
        p = []
        for i in range(0, N, self.gpu_batch):
            p.append(self.model(stack[i:min(i + self.gpu_batch, N)]))
        p = torch.cat(p)
        # Number of classes
        CL = p.size(1)

        # p: (N, CL), p.data.transpose(): (CL, N)
        # self.masks.view(): (N, h * w), height and width is flattend 
        sal = torch.matmul(p.data.transpose(0, 1), self.masks.view(N, H * W))
        sal = sal.view((CL, H, W))
        sal = sal / N / self.p1
        return sal
    

In [ ]:
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
image_path_1 = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"
image_path_2 = "/kaggle/working/BirdNest-Balanced-Classification-3/test/2/BrokenBig-1-_bmp_jpg.rf.07d60d806118bbff3c0e6354e7782086.jpg"
img1 = Image.open(image_path_1)
img2 = Image.open(image_path_2)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img1)
plt.title("Image 1")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img2)
plt.title("Image 2")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.model.to(device)

# 2. Updated Robust Wrapper
def yolo_predict_wrapper(x):
    loaded_model.model.eval()
    x = x.to(device)
    results = loaded_model.model(x)
    
    # Unwrap lists/tuples to find the raw logit tensor
    logits = results
    while isinstance(logits, (list, tuple)):
        logits = logits[0]
        
    return F.softmax(logits, dim=1)

# 3. Prepare Image (Ensure this matches your training size)
input_size = (224, 224) 
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Standard ImageNet normalization (usually used by YOLO classification)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_tensor = preprocess(img1).unsqueeze(0).to(device)

# 4. Initialize RISE
explainer = RISE(yolo_predict_wrapper, input_size, gpu_batch=20)
explainer.generate_masks(N=5000, s=6, p1=0.2)

# 5. Run Prediction & Explanation
with torch.no_grad():
    predictions = yolo_predict_wrapper(img_tensor)
    conf, class_idx = torch.max(predictions, dim=1)
    class_idx = class_idx.item()
    
    # Get saliency maps
    saliency_maps = explainer(img_tensor)
    target_sal_map = saliency_maps[class_idx].cpu().numpy()

# 6. Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(img.resize(input_size))
ax[0].set_title(f"Original Image\nClass Index: {class_idx}")
ax[0].axis("off")

# Overlay
ax[1].imshow(img.resize(input_size))
ax[1].imshow(target_sal_map, cmap='jet', alpha=0.5)
ax[1].set_title(f"RISE Saliency Map\nConf: {conf.item():.2%}")
ax[1].axis("off")

plt.show()

In [ ]:
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.model.to(device)

# 2. Updated Robust Wrapper
def yolo_predict_wrapper(x):
    loaded_model.model.eval()
    x = x.to(device)
    results = loaded_model.model(x)
    
    # Unwrap lists/tuples to find the raw logit tensor
    logits = results
    while isinstance(logits, (list, tuple)):
        logits = logits[0]
        
    return F.softmax(logits, dim=1)

# 3. Prepare Image (Ensure this matches your training size)
input_size = (224, 224) 
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Standard ImageNet normalization (usually used by YOLO classification)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_tensor = preprocess(img).unsqueeze(0).to(device)

# 4. Initialize RISE
# N = number of masks 
# s = grid size (divides the image into s x s number of grids, larger s means more grids so finer cells)
# p1 = visibility probability (0.2 means only 20% of the image will be visible, 80% of the image will be covered with masks)  
explainer = RISE(yolo_predict_wrapper, input_size, gpu_batch=20)
explainer.generate_masks(N=10000, s=6, p1=0.2)

# 5. Run Prediction & Explanation
with torch.no_grad():
    predictions = yolo_predict_wrapper(img_tensor)
    conf, class_idx = torch.max(predictions, dim=1)
    class_idx = class_idx.item()
    
    # Get saliency maps
    saliency_maps = explainer(img_tensor)
    target_sal_map = saliency_maps[class_idx].cpu().numpy()

# 6. Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(img.resize(input_size))
ax[0].set_title(f"Original Image\nClass Index: {class_idx}")
ax[0].axis("off")

# Overlay
ax[1].imshow(img.resize(input_size))
ax[1].imshow(target_sal_map, cmap='jet', alpha=0.5)
ax[1].set_title(f"RISE Saliency Map\nConf: {conf.item():.2%}")
ax[1].axis("off")

plt.show()

# RISE Part 2

In [ ]:
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.model.to(device)

# 2. Wrapper function
def yolo_predict_wrapper(x):
    loaded_model.model.eval()
    x = x.to(device)
    results = loaded_model.model(x)
    logits = results
    while isinstance(logits, (list, tuple)):
        logits = logits[0]
    return F.softmax(logits, dim=1)

# 3. Preparation
input_size = (224, 224) 
preprocess = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Process both images into tensors
img_tensor1 = preprocess(img1).unsqueeze(0).to(device)
img_tensor2 = preprocess(img2).unsqueeze(0).to(device)

# 4. Initialize Explainer (Adjust s and p1 for InvRISE or RISE preference)
# Using s=8 and p1=0.5 for a balance between RISE and InvRISE logic
explainer = RISE(yolo_predict_wrapper, input_size, gpu_batch=20)
explainer.generate_masks(N=5000, s=6, p1=0.2)

# 5. Run Prediction & Explanation for both
def get_explanation(img_t):
    with torch.no_grad():
        preds = yolo_predict_wrapper(img_t)
        conf, class_idx = torch.max(preds, dim=1)
        c_idx = class_idx.item()
        
        # Saliency map is a weighted average of masks normalized by expectation [cite: 181, 182]
        s_maps = explainer(img_t)
        target_map = s_maps[c_idx].cpu().numpy()
        return target_map, conf.item(), c_idx

map1, conf1, class1 = get_explanation(img_tensor1)
map2, conf2, class2 = get_explanation(img_tensor2)

# 6. Final Visualization (2x2 Grid)
fig, ax = plt.subplots(2, 2, figsize=(12, 10))

# Image 1
ax[0, 0].imshow(img1.resize(input_size))
ax[0, 0].set_title(f"Image 1 (Class {class1})")
ax[0, 0].axis("off")

ax[0, 1].imshow(img1.resize(input_size))
ax[0, 1].imshow(map1, cmap='jet', alpha=0.5)
ax[0, 1].set_title(f"Saliency Map 1\nConf: {conf1:.2%}")
ax[0, 1].axis("off")

# Image 2
ax[1, 0].imshow(img2.resize(input_size))
ax[1, 0].set_title(f"Image 2 (Class {class2})")
ax[1, 0].axis("off")

ax[1, 1].imshow(img2.resize(input_size))
ax[1, 1].imshow(map2, cmap='jet', alpha=0.5)
ax[1, 1].set_title(f"Saliency Map 2\nConf: {conf2:.2%}")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()

# InvRISE

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from skimage.transform import resize
from tqdm import tqdm


class InvRISE(nn.Module):
    def __init__(self, model, input_size, gpu_batch=100):
        super(InvRISE, self).__init__()
        self.model = model
        self.input_size = input_size
        self.gpu_batch = gpu_batch

    def generate_masks(self, N, s, p1, savepath='masks.npy'):
        cell_size = np.ceil(np.array(self.input_size) / s)
        up_size = (s + 1) * cell_size

        grid = np.random.rand(N, s, s) < p1
        grid = grid.astype('float32')

        self.masks = np.empty((N, *self.input_size))

        for i in tqdm(range(N), desc='Generating filters'):
            # Random shifts
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            # Linear upsampling and cropping
            self.masks[i, :, :] = resize(grid[i], up_size, order=1, mode='reflect',
                                         anti_aliasing=False)[x:x + self.input_size[0], y:y + self.input_size[1]]
        self.masks = self.masks.reshape(-1, 1, *self.input_size)
        np.save(savepath, self.masks)
        self.masks = torch.from_numpy(self.masks).float()
        self.masks = self.masks.cuda()
        self.N = N
        self.p1 = p1

    def load_masks(self, filepath):
        self.masks = np.load(filepath)
        self.masks = torch.from_numpy(self.masks).float().cuda()
        self.N = self.masks.shape[0]

    def forward(self, x):
        N = self.N
        _, _, H, W = x.size()
        # Apply array of filters to the image
        stack = torch.mul(self.masks, x.data)

        # p = nn.Softmax(dim=1)(model(stack)) processed in batches
        p = []
        for i in range(0, N, self.gpu_batch):
            p.append(self.model(stack[i:min(i + self.gpu_batch, N)]))
        p = torch.cat(p)
        # Number of classes
        CL = p.size(1)

        # p: (N, CL), p.data.transpose(): (CL, N)
        # self.masks.view(): (N, h * w), height and width is flattend 
        sal = torch.matmul((1 - p).data.transpose(0, 1), (1 - self.masks).view(N, H * W))
        sal = sal.view((CL, H, W))
        sal = sal / N / (1 - self.p1)
        return sal
    

In [ ]:
loaded_model = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")
image_path_1 = "/kaggle/working/BirdNest-Balanced-Classification-3/test/0/SizeA-15-_bmp_jpg.rf.7e3401521af6ebd6b1c382f907f5a4a7.jpg"
image_path_2 = "/kaggle/working/BirdNest-Balanced-Classification-3/test/2/BrokenBig-1-_bmp_jpg.rf.07d60d806118bbff3c0e6354e7782086.jpg"
img1 = Image.open(image_path_1)
img2 = Image.open(image_path_2)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img1)
plt.title("Image 1")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img2)
plt.title("Image 2")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.model.to(device)

# 2. Updated Robust Wrapper
def yolo_predict_wrapper(x):
    loaded_model.model.eval()
    x = x.to(device)
    results = loaded_model.model(x)
    
    # Unwrap lists/tuples to find the raw logit tensor
    logits = results
    while isinstance(logits, (list, tuple)):
        logits = logits[0]
        
    return F.softmax(logits, dim=1)

# 3. Prepare Image (Ensure this matches your training size)
input_size = (224, 224) 
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Standard ImageNet normalization (usually used by YOLO classification)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_tensor = preprocess(img).unsqueeze(0).to(device)

# 4. Initialize InvRISE
# N = number of masks 
# s = grid size (divides the image into s x s number of grids, larger s means more grids so finer cells)
# p1 = visibility probability (0.2 means only 20% of the image will be visible, 80% of the image will be covered with masks)  
explainer = InvRISE(yolo_predict_wrapper, input_size, gpu_batch=20)
explainer.generate_masks(N=10000, s=6, p1=0.8)

# 5. Run Prediction & Explanation
with torch.no_grad():
    predictions = yolo_predict_wrapper(img_tensor)
    conf, class_idx = torch.max(predictions, dim=1)
    class_idx = class_idx.item()
    
    # Get saliency maps
    saliency_maps = explainer(img_tensor)
    target_sal_map = saliency_maps[class_idx].cpu().numpy()

# 6. Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(img.resize(input_size))
ax[0].set_title(f"Original Image\nClass Index: {class_idx}")
ax[0].axis("off")

# Overlay
ax[1].imshow(img.resize(input_size))
ax[1].imshow(target_sal_map, cmap='jet', alpha=0.5)
ax[1].set_title(f"InvRISE Saliency Map\nConf: {conf.item():.2%}")
ax[1].axis("off")

plt.show()

# InvRISE Part 2 

In [ ]:
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.model.to(device)

# 2. Updated Robust Wrapper (Ensuring softmax for probability weighting)
def yolo_predict_wrapper(x):
    loaded_model.model.eval()
    x = x.to(device)
    results = loaded_model.model(x)
    
    # Unwrap lists/tuples to find the raw logit tensor
    logits = results
    while isinstance(logits, (list, tuple)):
        logits = logits[0]
        
    return F.softmax(logits, dim=1)

# 3. Prepare Images
input_size = (224, 224) 
preprocess = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create tensors for both images
img_tensor1 = preprocess(img1).unsqueeze(0).to(device)
img_tensor2 = preprocess(img2).unsqueeze(0).to(device)

# 4. Initialize InvRISE
# Using p1=0.8 as in your prompt: This keeps 80% visible and obscures 20%
# This is better for InvRISE as it allows surgical "poking" of small impurities
explainer = InvRISE(yolo_predict_wrapper, input_size, gpu_batch=20)
explainer.generate_masks(N=10000, s=6, p1=0.8)

# 5. Run Prediction & Explanation for both images
def get_inv_explanation(img_t):
    with torch.no_grad():
        predictions = yolo_predict_wrapper(img_t)
        conf, class_idx = torch.max(predictions, dim=1)
        c_idx = class_idx.item()
        
        # Saliency map weighted by (1 - p) and negated masks (1 - m)
        saliency_maps = explainer(img_t)
        target_sal_map = saliency_maps[c_idx].cpu().numpy()
        return target_sal_map, conf.item(), c_idx

map1, conf1, class1 = get_inv_explanation(img_tensor1)
map2, conf2, class2 = get_inv_explanation(img_tensor2)

# 6. Visualization (2x2 Grid)
fig, ax = plt.subplots(2, 2, figsize=(12, 10))

# Image 1 (Size A)
ax[0, 0].imshow(img1.resize(input_size))
ax[0, 0].set_title(f"Image 1 (Class {class1})")
ax[0, 0].axis("off")

ax[0, 1].imshow(img1.resize(input_size))
ax[0, 1].imshow(map1, cmap='jet', alpha=0.5)
ax[0, 1].set_title(f"InvRISE Map 1\nConf: {conf1:.2%}")
ax[0, 1].axis("off")

# Image 2 (Broken Big)
ax[1, 0].imshow(img2.resize(input_size))
ax[1, 0].set_title(f"Image 2 (Class {class2})")
ax[1, 0].axis("off")

ax[1, 1].imshow(img2.resize(input_size))
ax[1, 1].imshow(map2, cmap='jet', alpha=0.5)
ax[1, 1].set_title(f"InvRISE Map 2\nConf: {conf2:.2%}")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()